<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/05_the_doom_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 — The doom loop

**The claim you should be able to make when you finish:** *"Agent failure is a
feedback loop, not a decay curve. It has a cliff, the cliff moves when I change
the harness, and I've watched it move."*

Lab 3 gave the optimistic bound: independent steps, `p ** n`, graceful decay.
Real agents are worse than that bound, and they are worse in a specific,
reproducible shape.

The mechanism has three parts and none of them are exotic:

1. A step fails. The failure — an error, a retry, a wrong result — is **appended
   to the transcript**, because the transcript is the state (lab 1).
2. A longer transcript is a worse transcript. Attention is finite; recall and
   instruction-following degrade as context fills. Call it context rot.
3. A worse transcript makes the next step more likely to fail. Go to 1.

That is positive feedback, and positive feedback does not decay gracefully. It
holds up, holds up, holds up, and then falls over — which is why every agent
reliability report contains the sentence "it worked fine in testing". Short
tasks never enter the loop.

**The failure is a property of the horizon, not the model.**

Thirty minutes. Everything here is a simulation that runs in milliseconds, so
you can sweep a thousand configurations and see the shape.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## A word on what this is and is not

`agentlab.sim` is a **simulator**, not a benchmark. The absolute numbers are made
up. The *shape* is the lesson, and the shape is what transfers.

Three parameters describe the agent, and two of them come straight out of your
own traces. The third — `rot` — is the one you have to measure yourself: run
your eval at 10K, 100K and 500K tokens of filled context and fit it. Everyone's
is different, and everyone's is worse than they expect.

In [ ]:
from agentlab.sim import Agent, Harness, Task, simulate

agent = Agent(p_step=0.95, p_tool_error=0.04, rot=0.45)
print(agent)
print()
print(simulate(agent=agent, task=Task(n_required=12), harness=Harness(), seed=3))
print(simulate(agent=agent, task=Task(n_required=30), harness=Harness(), seed=3))

Same agent, same seed. Twelve productive steps: done. Thirty: dead.

## 1. The cliff

Predict first: at 95% per step with mild context rot, how does success rate fall
as the task gets longer? Draw the curve in your head before running the sweep.

In [ ]:
from agentlab.sim import horizon_sweep

rows = horizon_sweep(lengths=range(2, 41, 2), n_runs=400)
print(f"{'steps':>6} {'measured':>10} {'independent bound':>19} {'gap':>9}")
for r in rows:
    bar = "#" * int(r["measured"] * 40)
    print(f"{r['n_required']:>6} {r['measured']:>10.1%} {r['independent_bound']:>18.1%} "
          f"{r['gap']:>+9.1%}  {bar}")

That is not a decay curve. It is a **cliff**, and it is somewhere around 24-28
steps for this configuration.

Two things in the gap column point in opposite directions, and both matter:

- **at short horizons the agent beats the bound** (gap is negative). The `p**n`
  model assumes one bad step kills the run. It does not — a real agent just takes
  another step. Recovery is real and the bound ignores it.
- **at long horizons it loses badly** (gap is positive), because the failures
  start feeding each other.

**The bound is wrong in both directions.** The crossing point is where your agent
lives or dies.

In [ ]:
crossing = next(r for r in rows if r["gap"] > 0)
print(f"the curves cross at about {crossing['n_required']} steps")
print(f"below that, recovery wins; above it, the feedback loop does.")

## 2. Watching one run die

The aggregate shows the cliff. A single run shows the mechanism.

In [ ]:
run = simulate(agent=agent, task=Task(n_required=30), harness=Harness(max_steps=45), seed=11)
print(run, "\n")
print(f"{'step':>5} {'outcome':<12} {'progress':>9} {'context':>9} {'P(step)':>9} {'streak':>7}")
for r in run.rows:
    flag = "  <- flailing" if r["confused"] else ""
    print(f"{r['step']:>5} {r['outcome']:<12} {r['progress']:>9} {r['context_tokens']:>9,} "
          f"{r['p_effective']:>9.1%} {r['consecutive_failures']:>7}{flag}")

Read down the `P(step)` column. It starts at 95% and falls monotonically — not
because the model changed, but because the context filled. Then a failure streak
hits the confusion threshold and it halves again. From there the run is
effectively over; it just has not stopped yet.

That last part is the expensive bit. **The run keeps spending after it is dead.**

In [ ]:
dead_from = next((r["step"] for r in run.rows if r["confused"]), None)
if dead_from:
    wasted = sum(1 for r in run.rows if r["step"] >= dead_from)
    print(f"flailing from step {dead_from}; {wasted} of {len(run.rows)} steps "
          f"({wasted / len(run.rows):.0%}) were spent after the run was effectively over.")
else:
    print("this seed never entered the confusion state — try another")

## 3. The lever table

Here is the payoff. Same model, same task, same seeds. **The only thing that
changes is your harness.**

In [ ]:
from agentlab.sim import lever_sweep

print(f"{'harness':<20} {'success':>9} {'steps':>7} {'input tokens':>14} {'peak ctx':>10}")
for r in lever_sweep(n_runs=400, n_required=24):
    print(f"{r['harness']:<20} {r['success_rate']:>9.1%} {r['mean_steps']:>7.1f} "
          f"{r['mean_input_tokens']:>14,} {r['peak_context']:>10,}")

Three things worth sitting with:

**Clearing old tool results takes success from 72% to 100% and halves the token
bill.** Not a tradeoff — strictly better on both axes. The failures were being
caused by the context, and the context was mostly stale tool output nobody was
going to look at again. This is the single highest-leverage line in the lab.

**The circuit breaker slightly *lowers* success.** It aborts runs that would have
recovered. That is not a bug in the breaker; it is what a breaker is. It is a
**cost control, not a quality lever**, and shipping it as the latter is a common
and confusing mistake — people add it expecting reliability, watch the pass rate
drop, and conclude the breaker is broken.

**Compaction works too, and costs more than clearing.** It preserves information
clearing would drop (lab 7 measures exactly how much), and you pay for that in
tokens and in a summarisation call.

## 4. Where the cliff moves

The interesting question is not "does clearing help" but "how far does it move
the cliff". Same sweep, two harnesses.

In [ ]:
managed = horizon_sweep(lengths=range(4, 61, 4), n_runs=300, harness=Harness(clear_after=6))
plain = {r["n_required"]: r["measured"] for r in horizon_sweep(lengths=range(4, 61, 4), n_runs=300)}

print(f"{'steps':>6} {'no management':>15} {'clear_after=6':>15}")
for r in managed:
    print(f"{r['n_required']:>6} {plain[r['n_required']]:>15.1%} {r['measured']:>15.1%}")

def cliff(rows):
    return next((n for n, rate in rows if rate < 0.5), None)

print(f"\nfalls below 50% at:  no management {cliff(sorted(plain.items()))} steps, "
      f"clear_after=6 {cliff([(r['n_required'], r['measured']) for r in managed])} steps")

## 5. Charting it

`plots.dashboard` takes either a simulated run's rows or the per-turn records of
a real `Trace`, because both carry the same keys. That is deliberate — one chart
function, two sources. If you can read one, you can read the other.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from agentlab.plots import dashboard, use_style

    use_style()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    dashboard(simulate(agent=agent, task=Task(n_required=12), harness=Harness(), seed=3).rows,
              ax=axes[0], title="12 steps — finishes")
    dashboard(simulate(agent=agent, task=Task(n_required=30), harness=Harness(), seed=3).rows,
              ax=axes[1], title="30 steps — the doom loop")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — pip install matplotlib to see the chart")

## 6. Detecting it live, in a real agent

None of this needs a simulator to detect in production. The signals are all in
the trajectory, they are all cheap, and none of them need a judge or a label:

| signal | what it means | where |
|---|---|---|
| `repeat_calls` rising | the agent is going in circles | `Trace.repeat_calls` |
| `longest_cycle` >= 2 | an A-B-A-B loop | `Trace.longest_cycle()` |
| context past ~50% of window | rot territory; measure yours | `Trace.context_high_water` |
| consecutive tool errors | the streak that precedes flailing | your `on_step` hook |
| `stop_reason == "max_steps"` | the run did not finish, it was stopped | `Trace.stop_reason` |

The last one deserves a policy: a run that ends on `max_steps` should never be
reported as a success, and it is worth alerting on the *rate* of those, because
it climbs before your pass rate falls.

In [ ]:
from agentlab.loop import ModelResponse, ScriptedModel, Tool, ToolRegistry, run as run_agent
from agentlab.loop import text_block, tool_use_block

tools = ToolRegistry([Tool("search", "Search the index.", fn=lambda **kw: "no results")])

def circling():
    return ScriptedModel([
        ModelResponse([tool_use_block(f"t{i}", "search", {"q": "widget" if i % 2 else "gadget"})],
                      "tool_use") for i in range(12)
    ] + [ModelResponse([text_block("giving up")], "end_turn")])

trace = run_agent(circling(), tools, "find the widget", max_steps=13)
print(f"repeat_calls:  {trace.repeat_calls}")
print(f"longest_cycle: {trace.longest_cycle()}")
print(f"stop_reason:   {trace.stop_reason}")
print("\nA cycle length of 2 with no progress is a loop. Break it in on_step (lab 1, §8).")

## What you can now say

- *"Agent failure is a feedback loop, not a decay curve — failures append to the
  transcript, and a longer transcript fails more."*
- *"There's a cliff, and it's a property of the horizon, not the model. Short
  tasks never enter the loop, which is why it worked in testing."*
- *"Clearing old tool results took a 24-step task from 72% to 100% *and* halved
  the bill. Nothing about the model changed."*
- *"A circuit breaker is a cost control, not a quality lever — it aborts runs
  that would have recovered."*
- *"Repeat calls and cycle length are free derailment signals. No judge, no
  labels."*

## Next

**[Lab 6](06_evals.ipynb)** — how you would know any of this happened.